# 10 · Text from documents — `lakelogic.extract_text`

One function turns a document into text. It picks the reader by file type and tells you **which reader read it**, **what it found** (text and tables), and — when there is nothing to read — **says so** instead of returning an empty string that looks like success.

| Type | Reader | Notes |
|---|---|---|
| PDF | `pdfplumber` | text **and tables** |
| Scanned PDF | `pdfplumber` + **RapidOCR** | pages without a text layer are OCR'd |
| Images (PNG, JPEG, …) | **RapidOCR** | printed words; no PyTorch |
| Word (.docx) | `python-docx` | paragraphs **and tables** |
| PowerPoint (.pptx) | `python-pptx` | one block per slide |
| Excel (.xlsx) | `openpyxl` | each sheet as a table |
| Text, Markdown, CSV, JSON, YAML, HTML, email | built-in | no install |

This notebook **builds a sample of each type, reads it back, and validates the result** against what was put in. Everything runs locally; nothing is sent anywhere.

In [1]:
# The document readers, installed into THIS notebook's Python if any are missing.
# Installed by name: the `lakelogic[documents]` extra only exists from the next LakeLogic release.
# uv-created environments have no pip, so uv is used when pip is absent.
import importlib.util
import shutil
import subprocess
import sys

packages = {
    "pdfplumber": "pdfplumber",
    "rapidocr_onnxruntime": "rapidocr-onnxruntime",
    "docx": "python-docx",
    "pptx": "python-pptx",
    "openpyxl": "openpyxl",
    "reportlab": "reportlab",
    "PIL": "pillow",
}
missing = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]
print("Python:", sys.executable)
if missing:
    print("Installing:", ", ".join(missing))
    if importlib.util.find_spec("pip"):
        cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    elif shutil.which("uv"):
        cmd = ["uv", "pip", "install", "--python", sys.executable, *missing]
    else:
        raise RuntimeError(f"This Python has no pip and uv is not on PATH. Install these yourself: {' '.join(missing)}")
    subprocess.run(cmd, check=True)

import lakelogic

if not hasattr(lakelogic, "extract_text"):
    raise RuntimeError(
        "This LakeLogic has no extract_text yet: use the local repo's environment, or LakeLogic newer than "
        f"{getattr(lakelogic, '__version__', '?')} once released."
    )
from lakelogic import extract_text

print("lakelogic", getattr(lakelogic, "__version__", "dev"), "from", lakelogic.__file__)

Python: C:\_Personal\_SaaS\lakelogic\.venv\Scripts\python.exe


lakelogic 1.80.3 from C:\_Personal\_SaaS\lakelogic\lakelogic\__init__.py


## 1 · Build one sample of each type

Each sample carries a known **code word** and known facts, so we can check that the reader got them out.

In [2]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

out = Path("document_demo")
out.mkdir(exist_ok=True)
CODE = "PELICAN 42"


def font(size=40):
    for name in ("arial.ttf", "DejaVuSans.ttf", "LiberationSans-Regular.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    return ImageFont.load_default()


def text_image(lines, width=1100):
    img = Image.new("RGB", (width, 110 * len(lines) + 80), "white")
    d = ImageDraw.Draw(img)
    for i, line in enumerate(lines):
        d.text((40, 40 + i * 110), line, fill="black", font=font())
    return img


# PDF with text and a table
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import Paragraph, SimpleDocTemplate, Table, TableStyle

doc = SimpleDocTemplate(str(out / "trip_report.pdf"), pagesize=A4)
table = Table([["city", "trips", "avg_fare"], ["London", "1240", "14.20"], ["Paris", "980", "12.75"]])
table.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.5, colors.black)]))
doc.build([Paragraph(f"RideFlow weekly trip report. Code word {CODE}.", getSampleStyleSheet()["Normal"]), table])

# Scanned PDF: an image of a page, no text layer
text_image(["Scanned invoice", f"Code word {CODE}", "Total due 42.00 GBP"]).save(
    out / "scanned_invoice.pdf", "PDF", resolution=150
)

# Image (a screenshot of an error)
text_image(["Pipeline failed: silver_trips", f"Code word {CODE}"]).save(out / "error_screenshot.png")

# Word with a table
import docx

d = docx.Document()
d.add_heading("Design note: silver trips", 1)
d.add_paragraph(f"The code word is {CODE}. Trips are keyed on trip_id.")
t = d.add_table(rows=3, cols=2)
for r, (a, b) in enumerate([("field", "type"), ("trip_id", "string"), ("fare", "decimal")]):
    t.cell(r, 0).text, t.cell(r, 1).text = a, b
d.save(out / "design_note.docx")

# PowerPoint, two slides
from pptx import Presentation

deck = Presentation()
for title, body in [("Q3 roadmap", "Ship Lakehouse Studio"), (f"Code word {CODE}", "Questions?")]:
    s = deck.slides.add_slide(deck.slide_layouts[1])
    s.shapes.title.text = title
    s.placeholders[1].text = body
deck.save(out / "roadmap.pptx")

# Excel, one sheet with data and one empty
import openpyxl

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Fares"
for row in [["city", "base_fare"], ["London", 3.5], ["Paris", 3.0]]:
    ws.append(row)
wb.create_sheet("Empty")
wb.save(out / "fares.xlsx")

# Plain text, HTML, email
(out / "notes.md").write_text(f"# Notes\n\nThe code word is {CODE}.\n", encoding="utf-8")
(out / "page.html").write_text(
    f"<html><head><script>track()</script></head><body><h1>Status</h1><p>Code word {CODE}</p></body></html>",
    encoding="utf-8",
)
(out / "incident.eml").write_text(
    f"From: ops@rideflow.io\nTo: data@rideflow.io\nSubject: Incident 117\n\nCode word {CODE}. Silver trips failed at 02:00.\n",
    encoding="utf-8",
)

sorted(p.name for p in out.iterdir())

['design_note.docx',
 'error_screenshot.png',
 'fares.xlsx',
 'incident.eml',
 'notes.md',
 'page.html',
 'roadmap.pptx',
 'scanned_invoice.pdf',
 'trip_report.pdf']

## 2 · Read each one, and validate what came back

For every file: which reader read it, what text and tables it returned, and a **check** against the facts we put in. OCR results are compared case-insensitively and allow OCR's spacing.

In [3]:
import re


def has(text, needle):
    # OCR may drop or add a space ("PELICAN42"): compare letters and digits only.
    norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
    return norm(needle) in norm(text)


checks = {
    "trip_report.pdf": lambda r: r.extractor == "pdfplumber"
    and has(r.text, CODE)
    and ["Paris", "980", "12.75"] in r.tables[0],
    "scanned_invoice.pdf": lambda r: has(r.text, CODE) and has(r.text, "42.00") and "OCR" in (r.note or ""),
    "error_screenshot.png": lambda r: r.extractor == "rapidocr" and has(r.text, CODE) and has(r.text, "silver_trips"),
    "design_note.docx": lambda r: r.extractor == "python-docx"
    and has(r.text, CODE)
    and r.tables == [[["field", "type"], ["trip_id", "string"], ["fare", "decimal"]]],
    "roadmap.pptx": lambda r: r.extractor == "python-pptx"
    and r.units == 2
    and "[Slide 2]" in r.text
    and has(r.text, CODE),
    "fares.xlsx": lambda r: r.extractor == "openpyxl"
    and r.units == 1
    and r.tables == [[["city", "base_fare"], ["London", "3.5"], ["Paris", "3"]]],  # Excel stores 3.0 as 3
    "notes.md": lambda r: r.extractor == "plain" and has(r.text, CODE),
    "page.html": lambda r: has(r.text, CODE) and "track()" not in r.text,
    "incident.eml": lambda r: "Subject: Incident 117" in r.text and has(r.text, CODE),
}

results = {}
for name, check in checks.items():
    r = extract_text(out / name)
    ok = bool(check(r))
    results[name] = (r, ok)
    print(
        f"{'PASS' if ok else 'FAIL'}  {name:<22} reader={r.extractor:<12} units={r.units}  tables={len(r.tables)}"
        + (f"  note: {r.note}" if r.note else "")
    )

PASS  trip_report.pdf        reader=pdfplumber   units=1  tables=1


PASS  scanned_invoice.pdf    reader=pdfplumber   units=1  tables=0  note: 1 scanned page(s) read with OCR (1 had text).


PASS  error_screenshot.png   reader=rapidocr     units=None  tables=0
PASS  design_note.docx       reader=python-docx  units=None  tables=1
PASS  roadmap.pptx           reader=python-pptx  units=2  tables=0
PASS  fares.xlsx             reader=openpyxl     units=1  tables=1
PASS  notes.md               reader=plain        units=None  tables=0
PASS  page.html              reader=html         units=None  tables=0
PASS  incident.eml           reader=email        units=None  tables=0


## 3 · Look at the output

`as_plain_text()` is what a person — or an assistant like Zeus — reads: the text, then each table as tab-separated rows.

In [4]:
for name in [
    "trip_report.pdf",
    "scanned_invoice.pdf",
    "error_screenshot.png",
    "design_note.docx",
    "roadmap.pptx",
    "fares.xlsx",
]:
    r, ok = results[name]
    print(f"── {name}  ({r.extractor}) ──")
    print(r.as_plain_text()[:400])
    print()

── trip_report.pdf  (pdfplumber) ──
RideFlow weekly trip report. Code word PELICAN 42.
city trips avg_fare
London 1240 14.20
Paris 980 12.75

[Table 1]
city	trips	avg_fare
London	1240	14.20
Paris	980	12.75

── scanned_invoice.pdf  (pdfplumber) ──
Scanned invoice
CodewordPELICAN42
Totaldue42.00GBP

── error_screenshot.png  (rapidocr) ──
Pipeline failed: silver_trips
Code word PELICAN 42

── design_note.docx  (python-docx) ──
Design note: silver trips
The code word is PELICAN 42. Trips are keyed on trip_id.

[Table 1]
field	type
trip_id	string
fare	decimal

── roadmap.pptx  (python-pptx) ──
[Slide 1]
Q3 roadmap
Ship Lakehouse Studio

[Slide 2]
Code word PELICAN 42
Questions?

── fares.xlsx  (openpyxl) ──
Sheets: Fares

[Table 1]
city	base_fare
London	3.5
Paris	3



## 4 · Tables come back as rows

PDF, Word and Excel tables are returned as lists of rows, ready for a DataFrame.

In [5]:
import pandas as pd

r, _ = results["trip_report.pdf"]
header, *rows = r.tables[0]
pd.DataFrame(rows, columns=header)

,city,trips,avg_fare
0,London,1240,14.20
1,Paris,980,12.75


## 5 · When there is nothing to read, it says so

A blank image is read, finds no words, and reports `no_text` with a reason — not an empty success. Old Office formats are refused with what to do instead.

In [6]:
from lakelogic.core.text_extraction import UnsupportedDocument

Image.new("RGB", (400, 300), "white").save(out / "blank.png")
blank = extract_text(out / "blank.png")
print("blank.png ->", "no_text:", blank.no_text, "|", blank.note)
assert blank.no_text

for legacy in ["old_spec.doc", "old_deck.ppt", "old_book.xls"]:
    (out / legacy).write_bytes(b"x")
    try:
        extract_text(out / legacy)
    except UnsupportedDocument as e:
        print(f"{legacy} -> refused: {e}")

blank.png -> no_text: True | No text found in this image (OCR reads printed words, not charts or pictures).
old_spec.doc -> refused: .doc is an older Office format that needs LibreOffice to read. Save it as .docx and try again.
old_deck.ppt -> refused: .ppt is an older Office format that needs LibreOffice to read. Save it as .pptx and try again.
old_book.xls -> refused: .xls is an older Office format that needs LibreOffice to read. Save it as .xlsx and try again.


## 6 · Summary

In [7]:
summary = pd.DataFrame(
    [
        {
            "file": n,
            "reader": r.extractor,
            "characters": len(r.text),
            "tables": len(r.tables),
            "valid": "PASS" if ok else "FAIL",
        }
        for n, (r, ok) in results.items()
    ]
)
print(f"{(summary.valid == 'PASS').sum()} of {len(summary)} document types validated")
assert (summary.valid == "PASS").all(), summary[summary.valid != "PASS"]
summary

9 of 9 document types validated


,file,reader,characters,tables,valid
0,trip_report.pdf,pdfplumber,104,1,PASS
1,scanned_invoice.pdf,pdfplumber,50,0,PASS
2,error_screenshot.png,rapidocr,50,0,PASS
3,design_note.docx,python-docx,82,1,PASS
4,roadmap.pptx,python-pptx,85,0,PASS
5,fares.xlsx,openpyxl,13,1,PASS
6,notes.md,plain,39,0,PASS
7,page.html,html,27,0,PASS
8,incident.eml,email,117,0,PASS
